# Machine Translation

In today's competition, your task is to implement the method described in the paper "Neural machine translation by jointly learning to align and translate". You will receive points for implementing and training the model, computing appropriate metrics, and conducting additional experiments and visualizations. During your work, you should ensure code clarity and readability, as well as the aesthetics of presented plots. Your implementation will be evaluated for compliance with the publication.

## Competition Rules

You must adhere to the following rules:

- You may not use the Internet. Exceptions are the OpenAI API, PyTorch documentation, and the Olympiad Google Classroom.
- You may not use Copilot or any other models that help write code, except for models from the GPT3.5 family.
- You may not use your own notes: both handwritten and files on the computer (including in particular code downloaded to the computer).
- You may not connect to computing resources other than Google Colab with T4 GPU.

## Task and scoring

You will train models on a dataset containing pairs of sentences in English and German (*parallel corpus*). Remember good coding practices and correct formatting, as this will affect your grade. Based on the attached paper, perform the following subtasks.

**Subtask 1: Model implementation (7 pts)**

In this section, we ask you to implement the method from the paper very precisely.

**Subtask 2: Model training (3 pts)**

Train the model on the provided dataset (dataloaders are already implemented in the starter code). During training, monitor both training and validation loss. Then create plots of these losses as a function of iteration. Evaluate the trained model on the test subset of the provided dataset using appropriate metrics. You can use the metrics used in the paper. Present examples of both correct and incorrect translations.

**Subtask 3: Attention visualization (1 pt)**

Present visualizations of learned attention maps on example interesting sentences. You can follow the plots from the paper.

**Subtask 4: Additional experiments (2 pts)**

If you have ideas for additional interesting experiments, you can get extra points for them. You may consider model modifications, ablations, and others.

## Notes
* You may change function and class signatures, as well as the code structure proposed by us below. However, remember good practices.

## Starter code

In [ ]:
# ! pip install datasets
# ! pip install evaluate
# ! pip install torchtext

# # Download the embedding models
# ! python -m spacy download en_core_web_sm 
# ! python -m spacy download de_core_news_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ------ --------------------------------- 2.1/12.8 MB 9.8 MB/s eta 0:00:02
     ----------- ---------------------------- 3.7/12.8 MB 8.7 MB/s eta 0:00:02
     ----------------- ---------------------- 5.5/12.8 MB 8.6 MB/s eta 0:00:01
     ----------------------- ---------------- 7.6/12.8 MB 8.9 MB/s eta 0:00:01
     ----------------------------- ---------- 9.4/12.8 MB 8.9 MB/s eta 0:00:01
     ----------------------------------- ---- 11.3/12.8 MB 8.8 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 8.7 MB/s  0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/14.6 MB ? eta -:--:--
     - -------------------------------------- 0.5/14.6 MB 16.4 MB/s eta 0:00:01
     ------ --------------------------------- 2.4/14.6 MB 9.6 MB/s eta 0:00:02
     ------------ --------------------------- 4.5/14.6 MB 9.6 MB/s eta 0:00:02
     ----------------- ---------------------- 6.3/14.6 MB 9.4 MB/s eta 0:00:01
     --------------------- ------------------ 7.9/14.6 MB 9.0 MB/s eta 0:00:01
     -------------------------- ------------- 9.7/14.6 MB 9.0 MB/s eta 0:00:01
     ----------------------------- ---------- 10.7/14.6 MB 8.5 MB/s eta 0:00:01
     ---------------------------------- ----- 12.6/14.6 MB 8.4 MB/s eta 0:00:01
     ---------------------------------------  14.4/14.6 MB 8.4 MB/s eta 0:00:01
     ---------------------------------------- 14.6/14.6 MB 8.2 MB/s  0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import random

import datasets
import evaluate
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import spacy
import torch
import torch.nn as nn

from collections import Counter, OrderedDict

class SimpleVocab:
    def __init__(self, specials=None, min_freq=1):
        self.specials = specials if specials else []
        self.min_freq = min_freq
        self.stoi = OrderedDict()
        self.itos = OrderedDict()
        self.default_idx = None
        
    def build(self, iterator):
        counter = Counter()
        for tokens in iterator:
            counter.update(tokens)
        
        # Add special tokens first
        for token in self.specials:
            if token not in self.stoi:
                self.stoi[token] = len(self.stoi)
                self.itos[len(self.itos)] = token
        
        # Add other tokens meeting min_freq
        for token, freq in counter.items():
            if freq >= self.min_freq and token not in self.stoi:
                self.stoi[token] = len(self.stoi)
                self.itos[len(self.itos)] = token
    
    def lookup_indices(self, tokens):
        default = self.default_idx if self.default_idx is not None else self.stoi.get('<unk>', 0)
        return [self.stoi.get(token, default) for token in tokens]
    
    def lookup_tokens(self, indices):
        default = '<unk>'
        return [self.itos.get(idx, default) for idx in indices]
    
    def __getitem__(self, token):
        default = self.default_idx if self.default_idx is not None else self.stoi.get('<unk>', 0)
        return self.stoi.get(token, default)
    
    def __len__(self):
        return len(self.stoi)
    
    def set_default_index(self, idx):
        self.default_idx = idx

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
######################### DO NOT CHANGE THIS CELL ##########################
seed = 1234

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

## Datasets

We have prepared dataloaders and tokenizers for you.

In [6]:
######################### DO NOT CHANGE THIS CELL ##########################
dataset = datasets.load_dataset("bentrevett/multi30k")

en_nlp = spacy.load("en_core_web_sm")
de_nlp = spacy.load("de_core_news_sm")

sos_token = "<sos>"
eos_token = "<eos>"


def tokenize_example(example, max_length=1000):
    en_tokens = [token.text.lower() for token in en_nlp.tokenizer(example["en"])][:max_length]
    de_tokens = [token.text.lower() for token in de_nlp.tokenizer(example["de"])][:max_length]

    en_tokens = [sos_token] + en_tokens + [eos_token]
    de_tokens = [sos_token] + de_tokens + [eos_token]
    return {"en_tokens": en_tokens, "de_tokens": de_tokens}


train_data = dataset["train"].map(tokenize_example)
valid_data = dataset["validation"].map(tokenize_example)
test_data = dataset["test"].map(tokenize_example)


Map:   0%|          | 0/29000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1014 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [7]:
######################### DO NOT CHANGE THIS CELL ##########################
min_freq = 2
unk_token = "<unk>"
pad_token = "<pad>"

special_tokens = [unk_token, pad_token, sos_token, eos_token]

en_vocab = SimpleVocab(specials=special_tokens, min_freq=min_freq)
en_vocab.build(train_data["en_tokens"])

de_vocab = SimpleVocab(specials=special_tokens, min_freq=min_freq)
de_vocab.build(train_data["de_tokens"])

assert en_vocab[unk_token] == de_vocab[unk_token]
assert en_vocab[pad_token] == de_vocab[pad_token]

unk_index = en_vocab[unk_token]
pad_index = en_vocab[pad_token]

en_vocab.set_default_index(unk_index)
de_vocab.set_default_index(unk_index)

In [8]:
######################### DO NOT CHANGE THIS CELL ##########################
def numericalize_example(example):
    en_ids = en_vocab.lookup_indices(example["en_tokens"])
    de_ids = de_vocab.lookup_indices(example["de_tokens"])
    return {"en_ids": en_ids, "de_ids": de_ids}

train_data = train_data.map(numericalize_example)
valid_data = valid_data.map(numericalize_example)
test_data = test_data.map(numericalize_example)

Map:   0%|          | 0/29000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1014 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [9]:
######################### DO NOT CHANGE THIS CELL ##########################
format_columns = ["en_ids", "de_ids"]

train_data = train_data.with_format(
    type="torch", columns=format_columns, output_all_columns=True
)

valid_data = valid_data.with_format(
    type="torch",
    columns=format_columns,
    output_all_columns=True,
)

test_data = test_data.with_format(
    type="torch",
    columns=format_columns,
    output_all_columns=True,
)

In [10]:
######################### DO NOT CHANGE THIS CELL ##########################
BATCH_SIZE = 128

def get_data_loader(dataset, batch_size, pad_index, shuffle=False):
    def collate_fn(batch):
        batch_en_ids = [example["en_ids"] for example in batch]
        batch_de_ids = [example["de_ids"] for example in batch]
        batch_en_ids = nn.utils.rnn.pad_sequence(batch_en_ids, padding_value=pad_index)
        batch_de_ids = nn.utils.rnn.pad_sequence(batch_de_ids, padding_value=pad_index)
        batch = {
            "en_ids": batch_en_ids,
            "de_ids": batch_de_ids,
        }
        return batch

    data_loader = torch.utils.data.DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        collate_fn=collate_fn,
        shuffle=shuffle,
    )
    return data_loader

train_data_loader = get_data_loader(train_data, BATCH_SIZE, pad_index, shuffle=True)
valid_data_loader = get_data_loader(valid_data, BATCH_SIZE, pad_index)
test_data_loader = get_data_loader(test_data, BATCH_SIZE, pad_index)

## Subtask 1: Model implementation

In [11]:
class Encoder(nn.Module):
    def __init__(self, input_dim, embedding_dim, encoder_hidden_dim, decoder_hidden_dim, dropout):
        super().__init__()
        self.embed = nn.Embedding(input_dim, embedding_dim)
        self.dropout = nn.Dropout(dropout)
        self.birnn = nn.GRU(embedding_dim, encoder_hidden_dim, bidirectional=True)
        self.fc = nn.Linear(2*encoder_hidden_dim, decoder_hidden_dim)

    def forward(self, src):
        embeddings = self.dropout(self.embed(src))
        x, h = self.birnn(embeddings)
        # x: [src_len, batch, 2*enc_hidden]
        # h:  [2, batch, enc_hidden]   → dim-0: [forward_final, backward_final]
        h = torch.tanh(self.fc(torch.cat(h[0], h[1], dim=1)))
        return x, h

In [12]:
class Attention(nn.Module):
    def __init__(self, encoder_hidden_dim, decoder_hidden_dim):
        super().__init__()
        self.w_a = nn.Linear(encoder_hidden_dim, encoder_hidden_dim)
        self.u = nn.Linear(encoder_hidden_dim, 2 * encoder_hidden_dim)
        self.v_a = nn.Linear(encoder_hidden_dim, 1)

    def forward(self, hidden, encoder_outputs):
        energy = self.v_a(torch.tanh(self.w_a(encoder_outputs).unsqueeze(1) + self.u(hidden).permute(1,2,0))).squeeze(2)
        return torch.softmax(energy, dim=1)

In [13]:
class Decoder(nn.Module):
    def __init__(
        self,
        output_dim,
        embedding_dim,
        encoder_hidden_dim,
        decoder_hidden_dim,
        dropout,
        attention,
        maxout_hidden_dim=256
    ):
        super().__init__()
        self.attention = attention
        self.embed = nn.Embedding(output_dim, embedding_dim)
        self.dropout = nn.Dropout(dropout)
        self.birnn = nn.GRU(embedding_dim + 2*encoder_hidden_dim, decoder_hidden_dim)

        self.U_o = nn.Linear(decoder_hidden_dim,      maxout_hidden_dim * 2)
        self.V_o = nn.Linear(embedding_dim,            maxout_hidden_dim * 2)
        self.C_o = nn.Linear(encoder_hidden_dim * 2,  maxout_hidden_dim * 2)
        self.W_o = nn.Linear(maxout_hidden_dim, output_dim)

    @staticmethod
    def _maxout(x: torch.Tensor) -> torch.Tensor:
        return x.view(x.size(0), -1, 2).max(-1).values

    def forward(self, input, hidden, encoder_outputs):
        """
        Args:
            input:           [batch]                        — y_{i-1} token ids
            hidden:          [batch, dec_hidden]            — s_{i-1}
            encoder_outputs: [src_len, batch, 2*enc_hidden] — all annotations h_j

        Returns:
            prediction:  [batch, output_dim]   — unnormalised logits for y_i
            hidden:      [batch, dec_hidden]   — updated state s_i
            attn_weights:[batch, src_len]      — alpha_{ij} for visualisation
        """

        # 1. Embed previous target token
        embedded = self.dropout(self.embedding(input.unsqueeze(0)))  # [1, batch, emb_dim]

        # 2. Attention weights alpha_{ij}  (Eq. 6)
        attn_weights = self.attention(hidden, encoder_outputs)        # [batch, src_len]

        # 3. Context vector c_i = Σ_j alpha_{ij} h_j  (Eq. 5)
        context = torch.bmm(
            attn_weights.unsqueeze(1),                               # [batch, 1, src_len]
            encoder_outputs.permute(1, 0, 2),                        # [batch, src_len, 2*enc_hidden]
        ).permute(1, 0, 2)                                            # [1, batch, 2*enc_hidden]

        # 4. GRU step: s_i = f(s_{i-1}, y_{i-1}, c_i)
        rnn_input = torch.cat((embedded, context), dim=2)            # [1, batch, emb+2*enc]
        output, hidden = self.rnn(rnn_input, hidden.unsqueeze(0))
        # output: [1, batch, dec_hidden]   hidden: [1, batch, dec_hidden]

        # Squeeze sequence-length dimension
        output   = output.squeeze(0)    # [batch, dec_hidden]
        context  = context.squeeze(0)   # [batch, 2*enc_hidden]
        embedded = embedded.squeeze(0)  # [batch, emb_dim]

        # 5. Deep output with maxout (Appendix A.2.2)
        t_tilde = self.U_o(output) + self.V_o(embedded) + self.C_o(context)  # [batch, 2*maxout]
        t_i = self._maxout(t_tilde)          # [batch, maxout_hidden_dim]
        prediction = self.W_o(t_i)           # [batch, output_dim]

        return prediction, hidden.squeeze(0), attn_weights

In [14]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg):
        batch_size    = trg.shape[1]
        trg_len       = trg.shape[0]
        src_len       = src.shape[0]
        output_dim    = self.decoder.W_o.out_features

        outputs    = torch.zeros(trg_len, batch_size, output_dim).to(src.device)
        attentions = torch.zeros(trg_len, batch_size, src_len).to(src.device)

        # Encode the source sentence
        encoder_outputs, hidden = self.encoder(src)

        # First decoder input: <sos> token (trg[0] for every sentence in batch)
        dec_input = trg[0, :]

        for t in range(1, trg_len):
            prediction, hidden, attention = self.decoder(dec_input, hidden, encoder_outputs)
            outputs[t]    = prediction
            attentions[t] = attention

            # Teacher forcing
            use_teacher = random.random() < 0.5
            dec_input = trg[t] if use_teacher else prediction.argmax(dim=1)

        return outputs, attentions

In [15]:
input_dim = len(de_vocab)
output_dim = len(en_vocab)
encoder_embedding_dim = 256
decoder_embedding_dim = 256
encoder_hidden_dim = 512
decoder_hidden_dim = 512
encoder_dropout = 0.5
decoder_dropout = 0.5

attention = Attention(encoder_hidden_dim, decoder_hidden_dim)
encoder   = Encoder(
    input_dim, encoder_embedding_dim,
    encoder_hidden_dim, decoder_hidden_dim, encoder_dropout,
)
decoder   = Decoder(
    output_dim, decoder_embedding_dim,
    encoder_hidden_dim, decoder_hidden_dim,
    decoder_dropout, attention
)
model = Seq2Seq(encoder, decoder).to(device)

## Subtask 2: Model training

In [4]:
# TODO: Write the training loop. Remember to collect training and validation statistics.

# TODO: Train the model on the provided dataset

In [ ]:
# TODO: Create loss function plots

In [ ]:
# TODO: Evaluate the model on the test set

In [ ]:
def translate_sentence(sentence, model):
    # TODO
    return en_tokens, de_tokens, attention


# TODO: Examples of translations

## Subtask 3: Attention visualization

In [ ]:
def plot_attention(sentence, translation, attention):
    # TODO


# TODO: Attention visualization

## Subtask 4: Additional experiments


In [ ]:
# TODO